In [3]:
import torch, torchvision
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

torch: 2.6.0+cu126
torchvision: 0.21.0+cu126
CUDA: True
GPU: NVIDIA H100 80GB HBM3


In [10]:
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset

from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    T5ForConditionalGeneration,
    T5TokenizerFast,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

PROJECT_ROOT= Path('.').resolve()
PROCESSED_DIR= PROJECT_ROOT / 'data' / 'processed'
OUTPUT_DIR= PROJECT_ROOT / 'checkpoints' / 'run3'
HP_DIR= PROJECT_ROOT / 'checkpoints' / 'hp_search'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HP_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE= PROCESSED_DIR / 'wikisql_t5_train.pt'
VAL_FILE= PROCESSED_DIR / 'wikisql_t5_val.pt'

MODEL_NAME= 't5-base'

#hyperparams (updated from run2)
LR= 3e-4    
BATCH_SIZE= 32
EPOCHS= 20      #was 10
WARMUP_STEPS= 1000    #was 500
MAX_GEN_LEN= 128
LORA_R= 32      
LORA_ALPHA= 64      
LORA_MODULES= ['q', 'k', 'v', 'o', 'wi', 'wo'] 

print('Project root:',PROJECT_ROOT)
print('Train file:', TRAIN_FILE)
print('Val file:',VAL_FILE)
print('Output dir:',OUTPUT_DIR)

Project root: /projects/e32706/sct1077
Train file: /projects/e32706/sct1077/data/processed/wikisql_t5_train.pt
Val file: /projects/e32706/sct1077/data/processed/wikisql_t5_val.pt
Output dir: /projects/e32706/sct1077/checkpoints/run3


In [11]:
class TensorDictDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings
        self.length = encodings['input_ids'].size(0)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        return {
            'input_ids':self.encodings['input_ids'][idx],
            'attention_mask':self.encodings['attention_mask'][idx],
            'labels':self.encodings['labels'][idx],
        }


def load_tensor_dict(path: Path):
    if not path.exists():
        raise FileNotFoundError(f'Missing preprocessed file: {path}')
    data = torch.load(path, weights_only=False)
    required = {'input_ids', 'attention_mask', 'labels'}
    if not required.issubset(set(data.keys())):
        raise ValueError(f'{path} missing keys. Expected: {required}')
    return data


train_enc=load_tensor_dict(TRAIN_FILE)
val_enc=load_tensor_dict(VAL_FILE)

train_dataset=TensorDictDataset(train_enc)
val_dataset=TensorDictDataset(val_enc)

print('train examples:',len(train_dataset))
print('validation examples:', len(val_dataset))

train examples: 56355
validation examples: 8421


In [12]:
tokenizer = T5TokenizerFast.from_pretrained(MODEL_NAME)
def build_model(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1, target_modules=LORA_MODULES):
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=r,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=lora_dropout,
    )
    base  = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
    model = get_peft_model(base, lora_config)
    return model

build_model().print_trainable_parameters()

Loading weights: 100%|██████████| 257/257 [00:00<00:00, 8485.55it/s]


trainable params: 12,976,128 || all params: 235,879,680 || trainable%: 5.5012


In [13]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions= predictions[0]

    if predictions.ndim == 3:
        predictions = np.argmax(predictions, axis=-1)

    pad_id= tokenizer.pad_token_id
    vocab_size= tokenizer.vocab_size

    predictions= np.asarray(predictions, dtype=np.int64)
    labels= np.asarray(labels,dtype=np.int64)

    predictions= np.where(predictions== -100, pad_id, predictions)
    labels= np.where(labels== -100, pad_id, labels)

    predictions= np.clip(predictions, 0, vocab_size - 1)
    labels= np.clip(labels, 0, vocab_size - 1)

    decoded_preds= tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels= tokenizer.batch_decode(labels,skip_special_tokens=True)

    exact_match = sum(
        pred.strip() == label.strip()
        for pred, label in zip(decoded_preds, decoded_labels)
    ) / max(1, len(decoded_preds))

    return {'exact_match': exact_match}

#### Hyperband Search (Optuna)

Searches over LR, batch size, LoRA rank, dropout, and warmup steps.

In [31]:
!pip install optuna

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 60.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.9/613.9 kB 13.1 MB/s  0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [alembic]my]  WARNING: The script alembic is installed in '/home/sct1077/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [optuna]  WARNING: The script optuna is installed in '/home/sct1077/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [optuna]


In [ ]:
import optuna

_current_r = {'value': LORA_R}

def model_init_hp():
    return build_model(
        r= _current_r['value'],
        lora_alpha = _current_r['value'] * 2,
    )

def optuna_hp_space(trial):
    r = trial.suggest_categorical('lora_r', [16, 32, 64])
    _current_r['value'] = r
    return {
        'learning_rate':trial.suggest_float('learning_rate', 1e-4, 5e-4, log=True),
        'per_device_train_batch_size': trial.suggest_categorical('per_device_train_batch_size', [16, 32]),
        'weight_decay':trial.suggest_float('weight_decay', 0.0, 0.1),
        'warmup_steps':trial.suggest_int('warmup_steps', 200, 1000, step=200),
    }

hp_args = Seq2SeqTrainingArguments(
    output_dir= str(HP_DIR),
    num_train_epochs= 6,#max per trial; Hyperband prunes most at epoch 2
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    bf16= torch.cuda.is_available(),
    fp16= False,
    eval_strategy= 'epoch',
    save_strategy= 'no',
    predict_with_generate= False,    
    logging_steps= 200,
    report_to= 'none',
)

hp_trainer = Seq2SeqTrainer(
    model_init= model_init_hp,
    args= hp_args,
    train_dataset= train_dataset,
    eval_dataset= val_dataset,
    processing_class = tokenizer,
    compute_metrics= None,               
)

best_run = hp_trainer.hyperparameter_search(
    direction= 'minimize',#minimise val loss
    backend= 'optuna',
    hp_space= optuna_hp_space,
    n_trials= 12,#reduced from 20
    pruner= optuna.pruners.HyperbandPruner(
        min_resource= 1,#prune after just 1 epoch if bad
        max_resource= 6,
        reduction_factor= 3,
    ),
)

print('\nBest trial hyperparams:', best_run.hyperparameters)

LR= best_run.hyperparameters.get('learning_rate',LR)
BATCH_SIZE= best_run.hyperparameters.get('per_device_train_batch_size',BATCH_SIZE)
WARMUP_STEPS = best_run.hyperparameters.get('warmup_steps',WARMUP_STEPS)
LORA_R= best_run.hyperparameters.get('lora_r',LORA_R)
LORA_ALPHA= LORA_R * 2
print(f'Using: LR={LR}, BATCH={BATCH_SIZE}, WARMUP={WARMUP_STEPS}, r={LORA_R}, alpha={LORA_ALPHA}')

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 19152.41it/s]
[I 2026-05-31 20:53:57,702] A new study created in memory with name: no-name-384b8793-59f7-4e25-b773-ec4eb89c730f
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 19098.80it/s]
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss
1,0.216315,0.153641
2,0.130054,0.100627
3,0.109080,0.081853
4,0.097511,0.076390
5,0.090987,0.071788
6,0.087779,0.070594


[I 2026-05-31 21:18:10,653] Trial 0 finished with value: 0.07059390842914581 and parameters: {'lora_r': 64, 'learning_rate': 0.0001321254789525883, 'per_device_train_batch_size': 32, 'weight_decay': 0.04518647443261582, 'warmup_steps': 600}. Best is trial 0 with value: 0.07059390842914581.
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 16733.20it/s]
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss
1,0.126489,0.087951
2,0.089446,0.069727
3,0.076075,0.063688
4,0.068633,0.058304
5,0.060434,0.055150
6,0.056497,0.055087


[I 2026-05-31 21:42:29,203] Trial 1 finished with value: 0.055086612701416016 and parameters: {'lora_r': 16, 'learning_rate': 0.00027990064470202747, 'per_device_train_batch_size': 32, 'weight_decay': 0.02344960297773411, 'warmup_steps': 200}. Best is trial 1 with value: 0.055086612701416016.
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 14594.71it/s]
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


...

Epoch,Training Loss,Validation Loss
1,0.111164,0.101070


[I 2026-05-31 22:57:55,874] Trial 10 pruned. 
Loading weights: 100%|██████████| 257/257 [00:00<00:00, 17955.73it/s]
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss
1,0.114133,0.080044


[I 2026-05-31 23:03:51,141] Trial 11 pruned. 



Best trial hyperparams: {'lora_r': 32, 'learning_rate': 0.0004331450653013155, 'per_device_train_batch_size': 16, 'weight_decay': 0.057593095881329284, 'warmup_steps': 200}
Using: LR=0.0004331450653013155, BATCH=16, WARMUP=200, r=32, alpha=64


In [ ]:
#best trial hyperparams from hp search
LR = 0.0004331450653013155
BATCH_SIZE = 16
WARMUP_STEPS = 200
WEIGHT_DECAY = 0.057593095881329284   
LORA_R = 32
LORA_ALPHA = 64    
LORA_MODULES = ['q', 'k', 'v', 'o', 'wi', 'wo']
EPOCHS = 20        
MAX_GEN_LEN = 128


In [15]:
from transformers import DataCollatorForSeq2Seq

model = build_model(r=LORA_R, lora_alpha=LORA_ALPHA)
model.print_trainable_parameters()
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=False)

training_args = Seq2SeqTrainingArguments(
    output_dir= str(OUTPUT_DIR),
    num_train_epochs= EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size= 64,
    warmup_steps= WARMUP_STEPS,
    learning_rate= LR,
    fp16= False,
    bf16= torch.cuda.is_available(),
    eval_strategy= 'epoch',
    save_strategy= 'epoch',
    load_best_model_at_end= True,
    metric_for_best_model= 'exact_match',
    greater_is_better= True,
    predict_with_generate= True, 
    generation_max_length= MAX_GEN_LEN,
    label_smoothing_factor= 0.1,
    logging_steps= 100,
    save_total_limit= 2,
    report_to= 'none',
    weight_decay=WEIGHT_DECAY, 
)


trainer = Seq2SeqTrainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    processing_class = tokenizer,
    compute_metrics  = compute_metrics,
    data_collator    = data_collator,
)

print('Trainer initialised. Ready to train.')

Loading weights: 100%|██████████| 257/257 [00:00<00:00, 8730.35it/s]
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


trainable params: 12,976,128 || all params: 235,879,680 || trainable%: 5.5012
Trainer initialised. Ready to train.


In [ ]:
train_result = trainer.train()

print('\nTraining complete.')
print('Train metrics:', train_result.metrics)

eval_metrics = trainer.evaluate(max_length=MAX_GEN_LEN)
print('Validation metrics:', eval_metrics)

trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved model artifacts to', OUTPUT_DIR)

/home/sct1077/.local/lib/python3.13/site-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)


Epoch,Training Loss,Validation Loss,Exact Match
1,1.520083,1.489162,0.545422
2,1.497821,1.466702,0.608954
3,1.478878,1.461607,0.617504
4,1.472540,1.452659,0.647310
5,1.466016,1.453248,0.645054
6,1.464842,1.452017,0.648854
7,1.457605,1.448938,0.671298
8,1.452005,1.444825,0.671417
9,1.446431,1.444746,0.680442
10,1.447725,1.444325,0.679492


In [9]:
import matplotlib.pyplot as plt
import json
OUTPUT_DIR = Path('checkpoints/run3')
with open(OUTPUT_DIR / 'trainer_state.json') as f:
    history = json.load(f)['log_history']

train_epochs      = [entry['epoch'] for entry in history if 'loss' in entry]
train_losses      = [entry['loss']  for entry in history if 'loss' in entry]

eval_epochs       = [entry['epoch']                for entry in history if 'eval_loss' in entry]
eval_losses       = [entry['eval_loss']            for entry in history if 'eval_loss' in entry]
eval_exact_match  = [entry.get('eval_exact_match') for entry in history if 'eval_loss' in entry]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_epochs, train_losses, color='tab:blue', label='Train loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

if eval_epochs:
    axes[1].plot(eval_epochs, eval_losses, 'o-', color='tab:orange', label='Val loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss', color='tab:orange')
    axes[1].tick_params(axis='y', labelcolor='tab:orange')
    axes[1].grid(True, alpha=0.3)

    lines, labels = axes[1].get_legend_handles_labels()

    if any(m is not None for m in eval_exact_match):
        ax2 = axes[1].twinx()
        ax2.plot(eval_epochs, eval_exact_match, 's--', color='tab:green', label='Exact match')
        ax2.set_ylabel('Exact Match', color='tab:green')
        ax2.tick_params(axis='y', labelcolor='tab:green')
        lines2, labels2 = ax2.get_legend_handles_labels()
        lines+= lines2
        labels+= labels2

    axes[1].legend(lines, labels, loc='upper left')
    axes[1].set_title('Validation Loss & Exact Match')
else:
    axes[1].set_visible(False)

plt.tight_layout()
plt.show()

print(f'Final train loss: {train_losses[-1]:.4f}')
if eval_losses:
    print(f'Final val loss: {eval_losses[-1]:.4f}')
if eval_exact_match and eval_exact_match[-1] is not None:
    print(f'Final val exact match: {eval_exact_match[-1]:.4f}')

FileNotFoundError: [Errno 2] No such file or directory: 'checkpoints/run3/trainer_state.json'